# Croppy: train the on-phone leaf models (coffee + beans)
Runtime → Change runtime type → **T4 GPU**, then Run all (about 60–75 minutes for both models).

| Annex B.2 dataset | Role here |
|---|---|
| **BRACOL** (arabica leaves) | Trains the **coffee** model: healthy, miner, rust, phoma, cercospora |
| **iBean** (Makerere AI Lab, via TensorFlow Datasets `beans`) | Trains the **bean** model: angular leaf spot, bean rust, healthy. Real field photos from Uganda. |
| **PlantVillage** (via TFDS `plant_village`) + **PlantDoc** (GitHub) | Train an **`other`** class, *"this is not a coffee/bean leaf"*, so the model refuses instead of forcing a wrong label |
| RoCoLe + own photos (optional) | Field test for coffee, never used in training |

**Outputs:** `croppy_coffee_model.zip` → unzip into `app/model/`, and `croppy_beans_model.zip` → unzip into `app/model_beans/`.

In [ ]:
!pip -q install tensorflowjs
import os, json, shutil, zipfile, random, numpy as np, pandas as pd, tensorflow as tf, tensorflow_datasets as tfds
from pathlib import Path
from PIL import Image
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
SEED=42; random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
IMG=224; BATCH=32
POOL=Path("/content/pool")            # non-target leaves, one folder per source label
def save_img(dst, src=None, arr=None):
    """Re-encode every image as RGB JPEG (max 512 px). Skips corrupt files."""
    try:
        im=Image.open(src).convert("RGB") if src is not None else Image.fromarray(arr).convert("RGB")
        im.thumbnail((512,512)); dst.parent.mkdir(parents=True,exist_ok=True); im.save(dst,"JPEG",quality=90); return True
    except Exception: return False

## 1. Coffee leaves: BRACOL
Download the BRACOL leaf zip from Mendeley Data to Google Drive and set `ZIP_PATH`. The cell accepts either a `dataset.csv` with `predominant_stress`, or one folder per class.

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
ZIP_PATH="/content/drive/MyDrive/bracol_leaf.zip"   # <-- change me
RAW=Path("/content/raw"); shutil.rmtree(RAW,ignore_errors=True); RAW.mkdir()
with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(RAW)
COFFEE=Path("/content/data_coffee"); shutil.rmtree(COFFEE,ignore_errors=True)
STRESS={0:"healthy",1:"miner",2:"rust",3:"phoma",4:"cercospora"}
FOLDER_MAP={"health":"healthy","miner":"miner","rust":"rust","phoma":"phoma","cercospora":"cercospora","brown":"phoma"}
imgs={p.stem:p for p in RAW.rglob("*") if p.suffix.lower() in (".jpg",".jpeg",".png")}
csvs=[c for c in RAW.rglob("*.csv") if "predominant_stress" in pd.read_csv(c,nrows=1).columns]
if csvs:
    for _,r in pd.read_csv(csvs[0]).iterrows():
        p=imgs.get(str(r["id"]))
        if p is not None: save_img(COFFEE/STRESS[int(r["predominant_stress"])]/(p.stem+".jpg"),src=p)
else:
    for p in imgs.values():
        key=next((v for k,v in FOLDER_MAP.items() if k in p.parent.name.lower()),None)
        if key: save_img(COFFEE/key/(p.stem+".jpg"),src=p)
print({d.name:len(list(d.iterdir())) for d in COFFEE.iterdir()})

## 2. Bean leaves: iBean (TFDS `beans`, about 1,300 field photos from Uganda)

In [ ]:
BEANS=Path("/content/data_beans"); shutil.rmtree(BEANS,ignore_errors=True)
bds,binfo=tfds.load("beans",split="train+validation+test",as_supervised=True,with_info=True)
BMAP={"angular_leaf_spot":"bean_als","bean_rust":"bean_rust","healthy":"bean_healthy"}
bnames=binfo.features["label"].names
for i,(img,lab) in enumerate(tfds.as_numpy(bds)):
    save_img(BEANS/BMAP[bnames[lab]]/f"ib_{i}.jpg",arr=img)
print({d.name:len(list(d.iterdir())) for d in BEANS.iterdir()})

## 3. The "not this crop" pool: PlantVillage + PlantDoc
PlantVillage is studio images on plain backgrounds. PlantDoc is real field photos. Neither contains coffee. We use them only as **negative examples**, so the app can say *"this is not a coffee leaf"*. If the PlantVillage download fails, set `PV_N=0` and PlantDoc alone still works.

In [ ]:
shutil.rmtree(POOL,ignore_errors=True)
PV_N=2500
try:
    pv,pinfo=tfds.load("plant_village",split="train",as_supervised=True,with_info=True,shuffle_files=True)
    pnames=pinfo.features["label"].names
    for i,(img,lab) in enumerate(tfds.as_numpy(pv.shuffle(5000,seed=SEED).take(PV_N))):
        save_img(POOL/("pv__"+pnames[lab])/f"pv_{i}.jpg",arr=img)
except Exception as e: print("PlantVillage skipped:",e)
!git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset /content/plantdoc
for i,p in enumerate(Path("/content/plantdoc").rglob("*")):
    if p.suffix.lower() in (".jpg",".jpeg",".png"): save_img(POOL/("pd__"+p.parent.name)/f"pd_{i}.jpg",src=p)
labels=sorted(d.name for d in POOL.iterdir()); print(len(labels),"source labels,",sum(len(list(d.iterdir())) for d in POOL.iterdir()),"images")

In [ ]:
def add_other(target, n, exclude=(), extra=()):
    """Fill target/other with n images: from the pool (minus excluded labels) plus extra folders (e.g. the other crop)."""
    src=[p for d in POOL.iterdir() if not any(x in d.name.lower() for x in exclude) for p in d.iterdir()]
    for e in extra: src+=random.sample(list(Path(e).rglob("*.jpg")),min(n//4,len(list(Path(e).rglob("*.jpg")))))
    out=target/"other"; shutil.rmtree(out,ignore_errors=True); out.mkdir(parents=True)
    for i,p in enumerate(random.sample(src,min(n,len(src)))): shutil.copy(p,out/f"o_{i}.jpg")
    return len(list(out.iterdir()))

## 4. One training function for both crops (MobileNetV3-Small, head then fine-tune, threshold sweep, export)

In [ ]:
aug=tf.keras.Sequential([tf.keras.layers.RandomFlip("horizontal_and_vertical"),tf.keras.layers.RandomRotation(0.25),
    tf.keras.layers.RandomZoom((-0.2,0.2)),tf.keras.layers.RandomContrast(0.3),tf.keras.layers.RandomBrightness(0.3,value_range=(0,255))])
def load(path,label):
    x=tf.io.decode_jpeg(tf.io.read_file(path),channels=3); s=tf.reduce_min(tf.shape(x)[:2])
    x=tf.image.resize_with_crop_or_pad(x,s,s)                       # centre square, same as the app
    return tf.cast(tf.image.resize(x,(IMG,IMG)),tf.float32),label
def make_ds(items,train=False):
    d=tf.data.Dataset.from_tensor_slices(([a for a,_ in items],[b for _,b in items])).map(load,num_parallel_calls=tf.data.AUTOTUNE)
    if train: d=d.shuffle(1000,seed=SEED).map(lambda x,y:(tf.clip_by_value(aug(x,training=True),0,255),y),num_parallel_calls=tf.data.AUTOTUNE)
    return d.batch(BATCH).prefetch(tf.data.AUTOTUNE)
def probs_labels(model,d):
    P,Y=[],[]
    for x,y in d: P.append(model.predict(x,verbose=0)); Y.append(y.numpy())
    return np.concatenate(P),np.concatenate(Y)
def sweep(P,Y):
    rows=[]
    for t in np.arange(0.4,0.96,0.05):
        srt=np.sort(P,1); ok=(P.max(1)>=t)&((srt[:,-1]-srt[:,-2])>=0.15)
        rows.append((round(t,2),ok.mean(),(P.argmax(1)[ok]==Y[ok]).mean() if ok.any() else np.nan))
    return pd.DataFrame(rows,columns=["threshold","answered_share","accuracy_when_answered"])

def train_and_export(DATA, KEYS, name, crop, version):
    from sklearn.metrics import classification_report, confusion_matrix
    files=[(str(p),i) for i,k in enumerate(KEYS) for p in (DATA/k).iterdir()]
    random.shuffle(files); n=len(files); tr,va,te=files[:int(.7*n)],files[int(.7*n):int(.85*n)],files[int(.85*n):]
    trd,vad,ted=make_ds(tr,True),make_ds(va),make_ds(te)
    cw={i:len(tr)/(len(KEYS)*max(1,sum(1 for _,b in tr if b==i))) for i in range(len(KEYS))}
    base=tf.keras.applications.MobileNetV3Small(input_shape=(IMG,IMG,3),include_top=False,weights="imagenet",include_preprocessing=True)
    base.trainable=False
    inp=tf.keras.Input((IMG,IMG,3)); x=tf.keras.layers.GlobalAveragePooling2D()(base(inp,training=False))
    out=tf.keras.layers.Dense(len(KEYS),activation="softmax")(tf.keras.layers.Dropout(0.3)(x)); model=tf.keras.Model(inp,out)
    es=tf.keras.callbacks.EarlyStopping(patience=3,restore_best_weights=True)
    model.compile(tf.keras.optimizers.Adam(1e-3),"sparse_categorical_crossentropy",["accuracy"]); model.fit(trd,validation_data=vad,epochs=12,class_weight=cw,callbacks=[es])
    base.trainable=True
    for l in base.layers[:-40]: l.trainable=False
    model.compile(tf.keras.optimizers.Adam(1e-4),"sparse_categorical_crossentropy",["accuracy"]); model.fit(trd,validation_data=vad,epochs=8,class_weight=cw,callbacks=[es])
    Pv,Yv=probs_labels(model,vad); sv=sweep(Pv,Yv); print(sv)
    good=sv[sv.accuracy_when_answered>=0.95]; THR=float(good.threshold.min()) if len(good) else 0.85
    Pt,Yt=probs_labels(model,ted)
    print(f"\n=== {name} test | threshold {THR} ===\n",classification_report(Yt,Pt.argmax(1),target_names=KEYS,digits=3))
    print(confusion_matrix(Yt,Pt.argmax(1))); print(sweep(Pt,Yt).query("threshold==@THR"))
    ex=Path(f"/content/export_{name}"); shutil.rmtree(ex,ignore_errors=True)
    model.export(str(ex/"saved_model"))
    os.system(f"tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model --quantize_uint8 {ex/'saved_model'} {ex/'tfjs'}")
    json.dump({"crop":crop,"labels":KEYS,"input_size":IMG,"threshold":THR,"version":version,
               "notes":"Raw 0-255 RGB, centre-cropped square, softmax output. 'other' = not this crop (PlantVillage/PlantDoc negatives)."},
              open(ex/"tfjs"/"metadata.json","w"),indent=1)
    shutil.make_archive(f"/content/croppy_{name}_model","zip",ex/"tfjs")
    print(name,"model:",round(os.path.getsize(f"/content/croppy_{name}_model.zip")/1e6,2),"MB")
    return model,THR

## 5. Train the coffee model (5 diseases + `other`)
Negatives: PlantVillage/PlantDoc plus some iBean bean leaves, because her beans grow right below her coffee.

In [ ]:
COFFEE_KEYS=[k for k in ["healthy","miner","rust","phoma","cercospora"] if (COFFEE/k).exists() and any((COFFEE/k).iterdir())]
n_other=max(len(list((COFFEE/k).iterdir())) for k in COFFEE_KEYS)
print("other:",add_other(COFFEE,n_other,extra=[BEANS]))
coffee_model,COFFEE_THR=train_and_export(COFFEE,COFFEE_KEYS+["other"],"coffee","coffee","bracol+other-v2")

## 6. (Recommended) Field test on coffee photos the model has never seen
Put real field photos into `/content/drive/MyDrive/field_test/<key>/` (keys: healthy, rust, …). Use RoCoLe and **your own phone photos**. Also put a few non-coffee leaves in `field_test/other/`. The accuracy drop from the BRACOL test is your honest *"what our data does not cover"* evidence.

In [ ]:
FIELD=Path("/content/drive/MyDrive/field_test"); KEYS_C=COFFEE_KEYS+["other"]
if FIELD.exists():
    from sklearn.metrics import classification_report
    items=[]
    for d in FIELD.iterdir():
        if d.name in KEYS_C:
            for p in d.iterdir():
                q=Path("/content/ft")/d.name/(p.stem+".jpg")
                if save_img(q,src=p): items.append((str(q),KEYS_C.index(d.name)))
    Pf,Yf=probs_labels(coffee_model,make_ds(items)); present=sorted(set(Yf))
    print(classification_report(Yf,Pf.argmax(1),labels=present,target_names=[KEYS_C[i] for i in present],digits=3))
    print(sweep(Pf,Yf).query("threshold==@COFFEE_THR"))
else: print("No field set: say so in your video.")

## 7. Train the bean model (3 classes + `other`)
Negatives exclude soybean and other bean labels from the pool (too similar to label honestly), and add some coffee leaves.

In [ ]:
BEAN_KEYS=["bean_als","bean_rust","bean_healthy"]
n_other=max(len(list((BEANS/k).iterdir())) for k in BEAN_KEYS)
print("other:",add_other(BEANS,n_other,exclude=("soy","bean"),extra=[COFFEE/k for k in COFFEE_KEYS]))
bean_model,BEAN_THR=train_and_export(BEANS,BEAN_KEYS+["other"],"beans","beans","ibean+other-v1")

In [ ]:
from google.colab import files
for f in ["/content/croppy_coffee_model.zip","/content/croppy_beans_model.zip"]:
    if os.path.exists(f): files.download(f)

**Next:** unzip `croppy_coffee_model.zip` into `app/model/` and `croppy_beans_model.zip` into `app/model_beans/`. Alternatively, send the files to the phone and use *My farm → Load model files* (the metadata says which crop they belong to).

For the video, note each model's size, test accuracy, the share of photos answered at the chosen threshold, and how `other` performs.